[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/umangbagadi03/build-your-own-coding-agent/blob/main/notebooks/02_streaming.ipynb)

# 02. Streaming

**Part 1: The Loop** · about 15 minutes

In notebook 01 we waited for the whole answer before printing anything. For a coding agent that's a real problem: a long reply can leave you staring at a frozen terminal, wondering whether anything is happening.

Streaming fixes that. The model sends tiny pieces as it generates them, and you handle them one at a time.

**By the end you will have:**

- read a raw stream, byte format and all
- written the parser yourself
- measured the difference between *time to first token* and *total time*
- a `stream_chat()` function that returns a normal message at the end

In [1]:
import pathlib, urllib.request

if not pathlib.Path("llm_setup.py").exists():  # Colab opens one notebook at a time, so fetch the helper
    urllib.request.urlretrieve(
        "https://raw.githubusercontent.com/umangbagadi03/build-your-own-coding-agent/main/notebooks/llm_setup.py", "llm_setup.py"
    )

from llm_setup import connect

llm = connect()

OFFLINE MODE: a scripted local server, not a real model.
The HTTP calls are real. The answers are canned for these lessons.
Token counts are rough estimates and timings are simulated.
Set GROQ_API_KEY (free tier) or LLM_PROVIDER=ollama to use a real model.


In [2]:
import json
import time

import requests

url = f"{llm.base_url}/chat/completions"

headers = {"Content-Type": "application/json"}
if llm.api_key:
    headers["Authorization"] = f"Bearer {llm.api_key}"

PROMPT = [
    {"role": "user", "content": "In two sentences, explain why streaming makes chat apps feel faster."}
]

## 1. Look at the raw stream

Adding `"stream": True` to the payload changes the response format. We also pass `stream=True` to `requests`, which tells it not to wait for the whole body.

Let's print the first few raw lines exactly as they arrive.

In [3]:
payload = {"model": llm.model, "messages": PROMPT, "stream": True}

with requests.post(url, headers=headers, json=payload, stream=True, timeout=llm.timeout) as response:
    response.raise_for_status()
    print("Content-Type:", response.headers.get("Content-Type"))
    print()

    shown = 0
    for line in response.iter_lines():
        if line:  # events are separated by blank lines
            print(line.decode()[:150])
            shown += 1
        if shown == 4:
            break  # leaving the `with` block closes the connection, which cancels the request

Content-Type: text/event-stream



data: {"id": "offline-response", "object": "chat.completion.chunk", "model": "offline-script", "choices": [{"index": 0, "delta": {"role": "assistant",
data: {"id": "offline-response", "object": "chat.completion.chunk", "model": "offline-script", "choices": [{"index": 0, "delta": {"content": "Streamin
data: {"id": "offline-response", "object": "chat.completion.chunk", "model": "offline-script", "choices": [{"index": 0, "delta": {"content": "nds each
data: {"id": "offline-response", "object": "chat.completion.chunk", "model": "offline-script", "choices": [{"index": 0, "delta": {"content": "ce of th


This format is called **Server-Sent Events (SSE)**, and there's very little to it:

```
data: {"choices": [{"delta": {"content": "Hel"}}]}

data: {"choices": [{"delta": {"content": "lo"}}]}

data: [DONE]
```

- Each event is a line starting with `data: `, followed by a blank line.
- Each event holds a **delta**, meaning only the new bit of text rather than the whole message so far.
- The literal `[DONE]` ends the stream.

Also notice what we just did: `break` closed the connection partway through. That's how a "stop generating" button works.

## 2. Write the parser

It takes about ten lines. A generator suits this well, because the caller receives each piece as soon as it arrives.

In [4]:
def stream_text(messages):
    """Yield pieces of the assistant's text as they arrive."""
    payload = {"model": llm.model, "messages": messages, "stream": True}

    with requests.post(url, headers=headers, json=payload, stream=True, timeout=llm.timeout) as response:
        response.raise_for_status()
        for line in response.iter_lines():
            if not line.startswith(b"data: "):
                continue  # skip blank lines and SSE comments
            body = line[len(b"data: "):]
            if body == b"[DONE]":
                return
            chunk = json.loads(body)
            if chunk["choices"]:
                text = chunk["choices"][0]["delta"].get("content")
                if text:
                    yield text


for piece in stream_text(PROMPT):
    print(piece, end="", flush=True)
print()

Streaming se

nds each pie

ce of the an

swer as soon

 as it is ge

nerated, so 

... (13 outputs left out)

something is

 happening t

he whole tim

e.

## 3. Measure what users actually feel

Two numbers describe how fast a response feels:

- **Time to first token (TTFT):** how long the screen stays empty.
- **Total time:** how long until the answer is complete.

Streaming doesn't make the model finish sooner. It only changes *when you find out*. Let's measure both modes side by side. (In offline mode these timings are simulated; run this on a real model to get true numbers.)

In [5]:
# Without streaming: nothing to show until the very end.
started = time.perf_counter()
response = requests.post(
    url, headers=headers, json={"model": llm.model, "messages": PROMPT}, timeout=llm.timeout
)
response.raise_for_status()
blocking_total = time.perf_counter() - started

# With streaming: note when the first piece lands.
started = time.perf_counter()
first_token = None
for piece in stream_text(PROMPT):
    if first_token is None:
        first_token = time.perf_counter() - started
streaming_total = time.perf_counter() - started

print(f"{'':<12}{'first text visible':>20}{'complete':>12}")
print(f"{'blocking':<12}{blocking_total:>19.2f}s{blocking_total:>11.2f}s")
print(f"{'streaming':<12}{first_token:>19.2f}s{streaming_total:>11.2f}s")

              first text visible    complete
blocking                   1.23s       1.23s
streaming                  0.41s       1.35s


In [6]:
# Checkpoint: the first piece must arrive before the stream ends.
assert first_token is not None, "The stream produced no text."
assert first_token < streaming_total
print("Checkpoint passed.")

Checkpoint passed.


> **Real numbers can surprise you.** In one test on a small local model (Gemma 4 on Ollama), the table came out backwards: the blocking call finished in 38 seconds, while streaming showed its first text after 60 seconds. There are two likely reasons. Each request generates a different answer, so one comparison is noisy. And this model "reasons" before answering, which some providers send in a separate field that our parser ignores, so the screen stays blank during that time. Try running the cell a few times, and in exercise 2 try showing that reasoning field as well.

## 4. Get a normal message back

Printing text is only half the job. An agent also needs the complete message to add to its history, plus the finish reason and token usage. So let's collect everything while streaming.

`stream_options.include_usage` asks for one last chunk containing token counts. Its `choices` list is empty, which is why the parser above checks `if chunk["choices"]`. Not every provider sends it, so `usage` may come back as `None`.

In [7]:
def stream_chat(messages, on_text=None):
    """Stream a reply, call on_text for each piece, and return the finished result."""
    payload = {
        "model": llm.model,
        "messages": messages,
        "stream": True,
        "stream_options": {"include_usage": True},
    }
    pieces, finish_reason, usage = [], None, None

    with requests.post(url, headers=headers, json=payload, stream=True, timeout=llm.timeout) as response:
        response.raise_for_status()
        for line in response.iter_lines():
            if not line.startswith(b"data: "):
                continue
            body = line[len(b"data: "):]
            if body == b"[DONE]":
                break
            chunk = json.loads(body)
            usage = chunk.get("usage") or usage
            if not chunk["choices"]:
                continue
            choice = chunk["choices"][0]
            finish_reason = choice.get("finish_reason") or finish_reason
            text = choice["delta"].get("content")
            if text:
                pieces.append(text)
                if on_text:
                    on_text(text)

    return {
        "message": {"role": "assistant", "content": "".join(pieces)},
        "finish_reason": finish_reason,
        "usage": usage,
    }


seen = []
result = stream_chat(PROMPT, on_text=lambda text: (seen.append(text), print(text, end="", flush=True)))

print("\n")
print("Finish reason:", result["finish_reason"])
print("Usage:        ", result["usage"])

Streaming se

nds each pie

ce of the an

swer as soon

 as it is ge

nerated, so 

... (14 outputs left out)

 happening t

he whole tim

e.



Finish reason: stop
Usage:         {'prompt_tokens': 26, 'completion_tokens': 68, 'total_tokens': 94}


In [8]:
# Checkpoint: the pieces must add up to the final message, which is safe to put in the history.
assert "".join(seen) == result["message"]["content"]
assert result["message"]["role"] == "assistant"
assert result["finish_reason"] is not None, "The stream never reported why it stopped."
print("Checkpoint passed.")

Checkpoint passed.


We handle text here but ignore tool calls. Those stream as fragments too, and notebook 03 shows how to put them back together.

## Recap

- Streaming is SSE: `data: {json}` lines ending with `data: [DONE]`.
- Each chunk carries a **delta**, and you assemble the message yourself.
- Streaming can show text sooner, but it doesn't make the model finish faster, and a model that reasons first may still stay silent for a while.
- Closing the connection is how you cancel a response.

## Exercises

1. Build a "stop after N characters" feature using `break`, and check that the final text is cut off where you expect.
2. Print every field that appears in `delta` on a real model. If your provider sends a `reasoning` field, show it in a different style, and see how that changes the wait for the first visible text.
3. What happens if the connection dies halfway through? Stop the stream early in `stream_chat` and decide how an agent should treat the partial message.

**Next:** [03. Tool Calling](03_tool_calling.ipynb), the step that lets a model do things instead of just talking.